# 长音频四模型对比 (VAD)

使用 VAD 分段 + 四模型串行评估长录音。

In [5]:
import warnings, logging, os, re, json, time, gc, torch
logging.getLogger('root').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}, Device: {device}')

DATA_ROOT = os.path.expanduser('~/Desktop/hengdong_asr_trainset')
SV_CKPT = os.path.expanduser('~/Projects/Agent/model/sensevoice_lora/model.pt.best')
NANO_CKPT = os.path.expanduser('~/Projects/Agent/model/funasr_nano_v3/model.pt.best')

def clean_sensevoice_text(text):
    return re.sub(r'<\|[^|]*\|>', '', text).strip()

def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = list(range(len(s2) + 1))
    for c1 in s1:
        curr = [prev[0] + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

_PUNCT = re.compile(r'[\s\.,!?;:\-()\[\]{}\u3000\uff0c\u3002\uff01\uff1f\u3001\uff1b\uff1a\u201c\u201d\u2018\u2019\uff08\uff09\u3010\u3011\u300a\u300b\u2026\u2014]')
def norm_punct(text): return _PUNCT.sub('', text)

def free_memory():
    gc.collect()
    if torch.backends.mps.is_available(): torch.mps.empty_cache()

print('工具函数就绪')

PyTorch: 2.11.0, Device: mps
工具函数就绪


In [6]:
# 加载长音频数据
LONG_JSONL = os.path.join(DATA_ROOT, 'manifests/extra_long_talks.jsonl')
long_samples = []
with open(LONG_JSONL) as f:
    for line in f:
        item = json.loads(line)
        audio = item['audio_filepath']
        if audio.startswith('/mnt/data/'):
            audio = audio.replace('/mnt/data/hengdong_asr_trainset', DATA_ROOT)
        long_samples.append({
            'id': item['id'],
            'audio': audio,
            'text': item['text'],
            'duration': item['duration'],
            'speaker': item.get('speaker_id', 'unknown'),
        })
print(f'长音频: {len(long_samples)} 条')
for s in long_samples:
    exists = os.path.exists(s['audio'])
    print(f'  {s["speaker"]} | {s["duration"]/60:.1f}min | {"OK" if exists else "NOT FOUND"}')

长音频: 6 条
  talks_方言老女 | 27.1min | OK
  talks_方言老男 | 30.8min | OK
  talks_方言青女 | 23.3min | OK
  talks_方言青男 | 23.3min | OK
  dialogs_方言多人 | 16.7min | OK
  dialogs_方言多人 | 9.8min | OK


In [7]:
VAD_MODEL = 'iic/speech_fsmn_vad_zh-cn-16k-common-pytorch'
VAD_KWARGS = {'max_single_segment_time': 30000}

def eval_long_sensevoice(model, long_samples, label):
    """评估长音频 SenseVoice 模型 (VAD 内置)"""
    from funasr import AutoModel
    results = []
    start = time.time()
    for i, s in enumerate(long_samples):
        if not os.path.exists(s['audio']): continue
        print(f'  {label}: [{i+1}/{len(long_samples)}] {s["speaker"]} ({s["duration"]/60:.1f}min)')
        try:
            res = model.generate(input=s['audio'], language='auto', use_itn=True)
            pred_text = ''.join(clean_sensevoice_text(r['text']) for r in res if 'text' in r)
        except Exception as e:
            print(f'    Error: {e}')
            pred_text = ''
        exp_n = norm_punct(s['text'])
        pred_n = norm_punct(pred_text)
        cer = levenshtein(exp_n, pred_n) / max(len(exp_n), 1)
        results.append({
            'id': s['id'], 'speaker': s['speaker'], 'audio': s['audio'],
            'duration_min': s['duration'] / 60, 'cer': cer,
            'ref_len': len(exp_n), 'pred_len': len(pred_n),
            'expected': s['text'], 'predicted': pred_text,
        })
        print(f'    CER: {cer:.1%} | ref: {results[-1]["ref_len"]} | pred: {results[-1]["pred_len"]}')
    elapsed = time.time() - start
    avg_cer = sum(r['cer'] for r in results) / len(results) if results else 0
    return {'name': label, 'results': results, 'avg_cer': avg_cer, 'total': len(results), 'time': elapsed}

def eval_long_nano(asr_model, long_samples, label, direct=False):
    """评估长音频 Nano 模型
    - direct=False: 手动 VAD 分段 + 逐段推理
    - direct=True: 不分段，直接整音频传入模型（依赖模型内置 VAD）
    注意: Nano 不支持内置 VAD，需要手动 VAD 分段后逐段识别
    使用 soundfile 读取音频以避免 Mac 上 torchaudio 的兼容性问题
    """
    import soundfile as sf
    results = []
    start = time.time()
    with torch.inference_mode():
        for i, s in enumerate(long_samples):
            if not os.path.exists(s['audio']): continue
            mode_str = '不分段直接推理' if direct else 'VAD分段推理'
            print(f'  {label}: [{i+1}/{len(long_samples)}] {s["speaker"]} ({s["duration"]/60:.1f}min) [{mode_str}]')
            try:
                if direct:
                    # 方式1: 直接整音频传入（不分段）
                    res = asr_model.generate(input=s['audio'], language='auto', itn=True)
                    if res and 'text' in res[0]:
                        pred_text = res[0]['text']
                    else:
                        pred_text = ''
                    print(f'    直接推理完成, 输出长度: {len(pred_text)}')
                else:
                    # 方式2: VAD 分段后逐段推理
                    vad_model = AutoModel(model=VAD_MODEL, disable_update=True, device=device)
                    vad_res = vad_model.generate(input=s['audio'])
                    del vad_model
                    free_memory()
                    
                    segments = vad_res[0]['value'] if vad_res and 'value' in vad_res[0] else []
                    if not segments:
                        print(f'    VAD 未检测到语音段')
                        continue
                    print(f'    VAD 检测到 {len(segments)} 个语音段')

                    waveform, sr = sf.read(s['audio'], dtype='float32')
                    if waveform.ndim > 1:
                        waveform = waveform.squeeze()

                    pred_parts = []
                    for seg_idx, seg in enumerate(segments):
                        start_ms = int(seg[0])
                        end_ms = int(seg[1])
                        duration_ms = end_ms - start_ms
                        if duration_ms < 300:
                            continue

                        start_sample = int(start_ms * sr / 1000)
                        end_sample = int(end_ms * sr / 1000)
                        chunk = waveform[start_sample:end_sample]

                        if len(chunk) < sr * 0.3:
                            continue

                        import tempfile
                        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                            tmp_path = tmp.name
                        try:
                            sf.write(tmp_path, chunk, sr)
                            res = asr_model.generate(input=tmp_path, language='auto', itn=True)
                            if res and 'text' in res[0]:
                                pred_parts.append(res[0]['text'])
                        finally:
                            if os.path.exists(tmp_path):
                                os.unlink(tmp_path)

                        if (seg_idx + 1) % 100 == 0:
                            print(f'    [{seg_idx+1}/{len(segments)}] 已处理 {len(pred_parts)} 段')

                    pred_text = ''.join(pred_parts)
                    print(f'    识别完成: {len(pred_parts)}/{len(segments)} 段, 输出长度: {len(pred_text)}')

            except Exception as e:
                print(f'    Error: {e}')
                import traceback
                traceback.print_exc()
                pred_text = ''

            exp_n = norm_punct(s['text'])
            pred_n = norm_punct(pred_text)
            cer = levenshtein(exp_n, pred_n) / max(len(exp_n), 1)
            results.append({
                'id': s['id'], 'speaker': s['speaker'], 'audio': s['audio'],
                'duration_min': s['duration'] / 60, 'cer': cer,
                'ref_len': len(exp_n), 'pred_len': len(pred_n),
                'expected': s['text'], 'predicted': pred_text,
            })
            print(f'    CER: {cer:.1%} | ref: {results[-1]["ref_len"]} | pred: {results[-1]["pred_len"]}')

    elapsed = time.time() - start
    avg_cer = sum(r['cer'] for r in results) / len(results) if results else 0
    return {'name': label, 'results': results, 'avg_cer': avg_cer, 'total': len(results), 'time': elapsed}

print('长音频评估函数就绪')

长音频评估函数就绪


### 注意
长音频的参考文本是**摘要整理版**，而模型输出是**逐字转录版**，两者粒度不同。
因此 CER 偏高是正常现象，主要作为模型间横向对比参考。

### [1/4] SV-base (长音频)

In [8]:
from funasr import AutoModel

long_all_results = {}

print('[1/4] SV-base (长音频)')
long_model = AutoModel(
    model='iic/SenseVoiceSmall', disable_update=True, device=device,
    vad_model=VAD_MODEL, vad_kwargs=VAD_KWARGS,
)
long_all_results['SV-base'] = eval_long_sensevoice(long_model, long_samples, 'SV-base')
print(f'  平均CER={long_all_results["SV-base"]["avg_cer"]:.1%} | 耗时={long_all_results["SV-base"]["time"]:.1f}s')

del long_model
free_memory()

[1/4] SV-base (长音频)
funasr version: 1.3.1.
  SV-base: [1/6] talks_方言老女 (27.1min)


100%|██████████| 141/141 [00:02<00:00, 61.62it/s]
{'load_data': '0.000', 'extract_feat': '0.188', 'forward': '2.288', 'batch_size': '141', 'rtf': '0.010'}, : 100%|██████████| 141/141 [00:02<00:00, 61.62it/s]
rtf_avg: 0.010: 100%|██████████| 141/141 [00:02<00:00, 61.57it/s]                                                                                            

100%|██████████| 94/94 [00:01<00:00, 49.18it/s]
{'load_data': '0.000', 'extract_feat': '0.191', 'forward': '1.911', 'batch_size': '94', 'rtf': '0.008'}, : 100%|██████████| 94/94 [00:01<00:00, 49.18it/s]
rtf_avg: 0.008: 100%|██████████| 94/94 [00:01<00:00, 49.13it/s]                                                                                           

100%|██████████| 67/67 [00:01<00:00, 35.98it/s]
{'load_data': '0.000', 'extract_feat': '0.176', 'forward': '1.862', 'batch_size': '67', 'rtf': '0.007'}, : 100%|██████████| 67/67 [00:01<00:00, 35.98it/s]
rtf_avg: 0.007: 100%|██████████| 67/67 [00:01<00:00, 35.95it/s]        

    CER: 419.9% | ref: 909 | pred: 4108
  SV-base: [2/6] talks_方言老男 (30.8min)


100%|██████████| 179/179 [00:02<00:00, 82.71it/s]
{'load_data': '0.000', 'extract_feat': '0.237', 'forward': '2.164', 'batch_size': '179', 'rtf': '0.009'}, : 100%|██████████| 179/179 [00:02<00:00, 82.71it/s]
rtf_avg: 0.009: 100%|██████████| 179/179 [00:02<00:00, 82.64it/s]                                                                                            

100%|██████████| 128/128 [00:01<00:00, 69.99it/s]
{'load_data': '0.000', 'extract_feat': '0.164', 'forward': '1.829', 'batch_size': '128', 'rtf': '0.007'}, : 100%|██████████| 128/128 [00:01<00:00, 69.99it/s]
rtf_avg: 0.007: 100%|██████████| 128/128 [00:01<00:00, 69.92it/s]                                                                                            

100%|██████████| 90/90 [00:01<00:00, 51.90it/s]
{'load_data': '0.000', 'extract_feat': '0.152', 'forward': '1.734', 'batch_size': '90', 'rtf': '0.007'}, : 100%|██████████| 90/90 [00:01<00:00, 51.90it/s]
rtf_avg: 0.007: 100%|██████████| 90/90 [00:01<00:00, 51.85it/s]

    CER: 253.9% | ref: 1268 | pred: 3745
  SV-base: [3/6] talks_方言青女 (23.3min)


...

    CER: 285.4% | ref: 1515 | pred: 4638
  SV-base: [5/6] dialogs_方言多人 (16.7min)


100%|██████████| 28/28 [00:02<00:00, 11.67it/s]
{'load_data': '0.000', 'extract_feat': '0.077', 'forward': '2.399', 'batch_size': '28', 'rtf': '0.017'}, : 100%|██████████| 28/28 [00:02<00:00, 11.67it/s]
rtf_avg: 0.017: 100%|██████████| 28/28 [00:02<00:00, 11.66it/s]                                                                                           

100%|██████████| 14/14 [00:03<00:00,  4.52it/s]
{'load_data': '0.000', 'extract_feat': '0.144', 'forward': '3.099', 'batch_size': '14', 'rtf': '0.012'}, : 100%|██████████| 14/14 [00:03<00:00,  4.52it/s]
rtf_avg: 0.012: 100%|██████████| 14/14 [00:03<00:00,  4.51it/s]                                                                                           

100%|██████████| 10/10 [00:02<00:00,  4.80it/s]
{'load_data': '0.000', 'extract_feat': '0.170', 'forward': '2.082', 'batch_size': '10', 'rtf': '0.007'}, : 100%|██████████| 10/10 [00:02<00:00,  4.80it/s]
rtf_avg: 0.007: 100%|██████████| 10/10 [00:02<00:00,  4.80it/s]                

    CER: 450.5% | ref: 311 | pred: 1466
  SV-base: [6/6] dialogs_方言多人 (9.8min)


100%|██████████| 22/22 [00:01<00:00, 11.12it/s]
{'load_data': '0.000', 'extract_feat': '0.085', 'forward': '1.978', 'batch_size': '22', 'rtf': '0.013'}, : 100%|██████████| 22/22 [00:01<00:00, 11.12it/s]
rtf_avg: 0.013: 100%|██████████| 22/22 [00:01<00:00, 11.11it/s]                                                                                           

100%|██████████| 10/10 [00:05<00:00,  1.95it/s]
{'load_data': '0.000', 'extract_feat': '0.126', 'forward': '5.127', 'batch_size': '10', 'rtf': '0.020'}, : 100%|██████████| 10/10 [00:05<00:00,  1.95it/s]
rtf_avg: 0.020: 100%|██████████| 10/10 [00:05<00:00,  1.95it/s]                                                                                           

100%|██████████| 5/5 [00:01<00:00,  4.18it/s]
{'load_data': '0.000', 'extract_feat': '0.084', 'forward': '1.196', 'batch_size': '5', 'rtf': '0.008'}, : 100%|██████████| 5/5 [00:01<00:00,  4.18it/s]
rtf_avg: 0.008: 100%|██████████| 5/5 [00:01<00:00,  4.17it/s]                       

    CER: 576.5% | ref: 204 | pred: 1256
  平均CER=456.7% | 耗时=68.3s


### [2/4] SV-ft (微调) (长音频)

In [9]:
print('[2/4] SV-ft (长音频)')
long_model = AutoModel(
    model='iic/SenseVoiceSmall', lora_only=True, disable_update=True, device=device,
    vad_model=VAD_MODEL, vad_kwargs=VAD_KWARGS,
)
long_ckpt = torch.load(SV_CKPT, map_location='cpu', weights_only=False)
long_model.model.load_state_dict(long_ckpt['state_dict'], strict=False)
del long_ckpt
print('  微调权重已载入')

long_all_results['SV-ft'] = eval_long_sensevoice(long_model, long_samples, 'SV-ft')
print(f'  平均CER={long_all_results["SV-ft"]["avg_cer"]:.1%} | 耗时={long_all_results["SV-ft"]["time"]:.1f}s')

del long_model
free_memory()

[2/4] SV-ft (长音频)
funasr version: 1.3.1.
  微调权重已载入
  SV-ft: [1/6] talks_方言老女 (27.1min)


100%|██████████| 141/141 [00:01<00:00, 72.98it/s]
{'load_data': '0.000', 'extract_feat': '0.184', 'forward': '1.932', 'batch_size': '141', 'rtf': '0.009'}, : 100%|██████████| 141/141 [00:01<00:00, 72.98it/s]
rtf_avg: 0.009: 100%|██████████| 141/141 [00:01<00:00, 72.91it/s]                                                                                            

100%|██████████| 94/94 [00:01<00:00, 53.12it/s]
{'load_data': '0.000', 'extract_feat': '0.165', 'forward': '1.769', 'batch_size': '94', 'rtf': '0.007'}, : 100%|██████████| 94/94 [00:01<00:00, 53.12it/s]
rtf_avg: 0.007: 100%|██████████| 94/94 [00:01<00:00, 53.08it/s]                                                                                           

100%|██████████| 67/67 [00:01<00:00, 38.64it/s]
{'load_data': '0.000', 'extract_feat': '0.161', 'forward': '1.734', 'batch_size': '67', 'rtf': '0.007'}, : 100%|██████████| 67/67 [00:01<00:00, 38.64it/s]
rtf_avg: 0.007: 100%|██████████| 67/67 [00:01<00:00, 38.60it/s]        

    CER: 392.8% | ref: 909 | pred: 3983
  SV-ft: [2/6] talks_方言老男 (30.8min)


100%|██████████| 179/179 [00:02<00:00, 88.16it/s]
{'load_data': '0.000', 'extract_feat': '0.210', 'forward': '2.030', 'batch_size': '179', 'rtf': '0.008'}, : 100%|██████████| 179/179 [00:02<00:00, 88.16it/s]
rtf_avg: 0.008: 100%|██████████| 179/179 [00:02<00:00, 88.09it/s]                                                                                            

100%|██████████| 128/128 [00:01<00:00, 69.66it/s]
{'load_data': '0.000', 'extract_feat': '0.188', 'forward': '1.837', 'batch_size': '128', 'rtf': '0.007'}, : 100%|██████████| 128/128 [00:01<00:00, 69.66it/s]
rtf_avg: 0.007: 100%|██████████| 128/128 [00:01<00:00, 69.59it/s]                                                                                            

100%|██████████| 90/90 [00:01<00:00, 51.11it/s]
{'load_data': '0.000', 'extract_feat': '0.176', 'forward': '1.761', 'batch_size': '90', 'rtf': '0.007'}, : 100%|██████████| 90/90 [00:01<00:00, 51.11it/s]
rtf_avg: 0.007: 100%|██████████| 90/90 [00:01<00:00, 51.06it/s]

    CER: 219.6% | ref: 1268 | pred: 3624
  SV-ft: [3/6] talks_方言青女 (23.3min)


...

    CER: 272.7% | ref: 1515 | pred: 4497
  SV-ft: [5/6] dialogs_方言多人 (16.7min)


100%|██████████| 28/28 [00:02<00:00, 10.86it/s]
{'load_data': '0.000', 'extract_feat': '0.079', 'forward': '2.578', 'batch_size': '28', 'rtf': '0.019'}, : 100%|██████████| 28/28 [00:02<00:00, 10.86it/s]
rtf_avg: 0.019: 100%|██████████| 28/28 [00:02<00:00, 10.84it/s]                                                                                           

100%|██████████| 14/14 [00:03<00:00,  4.36it/s]
{'load_data': '0.000', 'extract_feat': '0.198', 'forward': '3.210', 'batch_size': '14', 'rtf': '0.012'}, : 100%|██████████| 14/14 [00:03<00:00,  4.36it/s]
rtf_avg: 0.012: 100%|██████████| 14/14 [00:03<00:00,  4.36it/s]                                                                                           

100%|██████████| 10/10 [00:02<00:00,  4.81it/s]
{'load_data': '0.000', 'extract_feat': '0.165', 'forward': '2.078', 'batch_size': '10', 'rtf': '0.007'}, : 100%|██████████| 10/10 [00:02<00:00,  4.81it/s]
rtf_avg: 0.007: 100%|██████████| 10/10 [00:02<00:00,  4.81it/s]                

    CER: 435.7% | ref: 311 | pred: 1445
  SV-ft: [6/6] dialogs_方言多人 (9.8min)


100%|██████████| 22/22 [00:02<00:00,  9.19it/s]
{'load_data': '0.000', 'extract_feat': '0.115', 'forward': '2.393', 'batch_size': '22', 'rtf': '0.015'}, : 100%|██████████| 22/22 [00:02<00:00,  9.19it/s]
rtf_avg: 0.015: 100%|██████████| 22/22 [00:02<00:00,  9.18it/s]                                                                                           

100%|██████████| 10/10 [00:05<00:00,  1.89it/s]
{'load_data': '0.000', 'extract_feat': '0.139', 'forward': '5.296', 'batch_size': '10', 'rtf': '0.021'}, : 100%|██████████| 10/10 [00:05<00:00,  1.89it/s]
rtf_avg: 0.021: 100%|██████████| 10/10 [00:05<00:00,  1.89it/s]                                                                                           

100%|██████████| 5/5 [00:01<00:00,  3.96it/s]
{'load_data': '0.000', 'extract_feat': '0.100', 'forward': '1.264', 'batch_size': '5', 'rtf': '0.008'}, : 100%|██████████| 5/5 [00:01<00:00,  3.96it/s]
rtf_avg: 0.008: 100%|██████████| 5/5 [00:01<00:00,  3.95it/s]                       

    CER: 683.3% | ref: 204 | pred: 1504
  平均CER=477.2% | 耗时=67.0s


### [3/4] Nano-base (长音频)

In [10]:
print('[3/4] Nano-base (长音频, VAD分段)')
nano_base = AutoModel(model='FunAudioLLM/Fun-ASR-Nano-2512', hub='ms', disable_update=True, device=device)

long_all_results['Nano-base'] = eval_long_nano(nano_base, long_samples, 'Nano-base', direct=False)
print(f'  平均CER={long_all_results["Nano-base"]["avg_cer"]:.1%} | 耗时={long_all_results["Nano-base"]["time"]:.1f}s')

del nano_base
free_memory()

[3/4] Nano-base (长音频, VAD分段)
funasr version: 1.3.1.
  Nano-base: [1/6] talks_方言老女 (27.1min) [VAD分段推理]
funasr version: 1.3.1.


rtf_avg: 0.198: 100%|██████████| 1/1 [00:01<00:00,  1.65s/it]                                                                                          


    VAD 检测到 373 个语音段


rtf_avg: 0.241: 100%|██████████| 1/1 [00:00<00:00,  2.03it/s]                                                                                          


    [100/373] 已处理 100 段


...

    识别完成: 46/46 段, 输出长度: 5579
    CER: 1696.8% | ref: 311 | pred: 5392
  Nano-base: [6/6] dialogs_方言多人 (9.8min) [VAD分段推理]
funasr version: 1.3.1.


rtf_avg: 0.037: 100%|██████████| 1/1 [00:01<00:00,  1.65s/it]                                                                                          


    VAD 检测到 30 个语音段


rtf_avg: 0.171: 100%|██████████| 1/1 [00:04<00:00,  4.44s/it]                                                                                          


    识别完成: 30/30 段, 输出长度: 3081
    CER: 1398.0% | ref: 204 | pred: 2956
  平均CER=900.6% | 耗时=1887.3s


### [4/4] Nano-ft (微调) (长音频)

In [11]:
print('[4/4] Nano-ft (长音频, VAD分段)')
nano_ft = AutoModel(
    model='FunAudioLLM/Fun-ASR-Nano-2512', hub='ms', disable_update=True, device=device,
    init_param=NANO_CKPT,
)
print('  微调权重已载入')

long_all_results['Nano-ft'] = eval_long_nano(nano_ft, long_samples, 'Nano-ft', direct=False)
print(f'  平均CER={long_all_results["Nano-ft"]["avg_cer"]:.1%} | 耗时={long_all_results["Nano-ft"]["time"]:.1f}s')

del nano_ft
free_memory()

[4/4] Nano-ft (长音频, VAD分段)
funasr version: 1.3.1.
  微调权重已载入
  Nano-ft: [1/6] talks_方言老女 (27.1min) [VAD分段推理]
funasr version: 1.3.1.


rtf_avg: 0.248: 100%|██████████| 1/1 [00:02<00:00,  2.07s/it]                                                                                          


    VAD 检测到 373 个语音段


rtf_avg: 0.291: 100%|██████████| 1/1 [00:00<00:00,  1.68it/s]                                                                                          


    [100/373] 已处理 100 段


...

    识别完成: 46/46 段, 输出长度: 8079
    CER: 2312.9% | ref: 311 | pred: 7292
  Nano-ft: [6/6] dialogs_方言多人 (9.8min) [VAD分段推理]
funasr version: 1.3.1.


rtf_avg: 0.031: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it]                                                                                          


    VAD 检测到 30 个语音段


rtf_avg: 0.144: 100%|██████████| 1/1 [00:03<00:00,  3.74s/it]                                                                                          


    识别完成: 30/30 段, 输出长度: 3240
    CER: 1490.2% | ref: 204 | pred: 3144
  平均CER=1081.5% | 耗时=2137.4s


### [5/8] Nano-base (长音频, 不分段直接推理)

In [12]:
print('[5/8] Nano-base (长音频, 不分段直接推理)')
nano_base_direct = AutoModel(model='FunAudioLLM/Fun-ASR-Nano-2512', hub='ms', disable_update=True, device=device)

long_all_results['Nano-base-direct'] = eval_long_nano(nano_base_direct, long_samples, 'Nano-base-direct', direct=True)
print(f'  平均CER={long_all_results["Nano-base-direct"]["avg_cer"]:.1%} | 耗时={long_all_results["Nano-base-direct"]["time"]:.1f}s')

del nano_base_direct
free_memory()

[5/8] Nano-base (长音频, 不分段直接推理)
funasr version: 1.3.1.
  Nano-base-direct: [1/6] talks_方言老女 (27.1min) [不分段直接推理]


  0%|          | 0/1 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/var/folders/t3/g7rz1lhs145bf294dswkj5gh0000gn/T/ipykernel_27322/748423647.py", line 50, in eval_long_nano
    res = asr_model.generate(input=s['audio'], language='auto', itn=True)
  File "/Users/fanhua/Projects/Agent/local/.venv/lib/python3.14/site-packages/funasr/auto/auto_model.py", line 324, in generate
    return self.inference(
           ~~~~~~~~~~~~~~^
        input, input_len=input_len, progress_callback=progress_callback, **cfg
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/fanhua/Projects/Agent/local/.venv/lib/python3.14/site-packages/funasr/auto/auto_model.py", line 381, in inference
    res = model.inference(**batch, **kwargs)
  File "/Users/fanhua/Projects/Agent/local/.venv/lib/python3.14/site-packages/funasr/models/fun_asr_nano/model.py", line 603, in inference
    return self.inference_llm(
           ~~~~~~~~~~~~~~~~~~^
        data

    Error: Invalid buffer size: 10.98 GiB
    CER: 100.0% | ref: 909 | pred: 0
  Nano-base-direct: [2/6] talks_方言老男 (30.8min) [不分段直接推理]



  0%|          | 0/1 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/var/folders/t3/g7rz1lhs145bf294dswkj5gh0000gn/T/ipykernel_27322/748423647.py", line 50, in eval_long_nano
    res = asr_model.generate(input=s['audio'], language='auto', itn=True)
  File "/Users/fanhua/Projects/Agent/local/.venv/lib/python3.14/site-packages/funasr/auto/auto_model.py", line 324, in generate
    return self.inference(
           ~~~~~~~~~~~~~~^
        input, input_len=input_len, progress_callback=progress_callback, **cfg
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/fanhua/Projects/Agent/local/.venv/lib/python3.14/site-packages/funasr/auto/auto_model.py", line 381, in inference
    res = model.inference(**batch, **kwargs)
  File "/Users/fanhua/Projects/Agent/local/.venv/lib/python3.14/site-packages/funasr/models/fun_asr_nano/model.py", line 603, in inference
    return self.inference_llm(
           ~~~~~~~~~~~~~~~~~~^
        dat

    Error: Invalid buffer size: 14.10 GiB
    CER: 100.0% | ref: 1268 | pred: 0
  Nano-base-direct: [3/6] talks_方言青女 (23.3min) [不分段直接推理]


...

    Error: MPS backend out of memory (MPS allocated: 16.50 GiB, other allocations: 39.41 MiB, max allowed: 18.13 GiB). Tried to allocate 4.15 GiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).


Traceback (most recent call last):
  File "/var/folders/t3/g7rz1lhs145bf294dswkj5gh0000gn/T/ipykernel_27322/748423647.py", line 50, in eval_long_nano
    res = asr_model.generate(input=s['audio'], language='auto', itn=True)
  File "/Users/fanhua/Projects/Agent/local/.venv/lib/python3.14/site-packages/funasr/auto/auto_model.py", line 324, in generate
    return self.inference(
           ~~~~~~~~~~~~~~^
        input, input_len=input_len, progress_callback=progress_callback, **cfg
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/fanhua/Projects/Agent/local/.venv/lib/python3.14/site-packages/funasr/auto/auto_model.py", line 381, in inference
    res = model.inference(**batch, **kwargs)
  File "/Users/fanhua/Projects/Agent/local/.venv/lib/python3.14/site-packages/funasr/models/fun_asr_nano/model.py", line 603, in inference
    return self.inference_llm(
           ~~~~~~~~~~~~~~~~~~^
        data_in,
        ^^^^^^^^
    ...<4 lines

    CER: 100.0% | ref: 311 | pred: 0
  Nano-base-direct: [6/6] dialogs_方言多人 (9.8min) [不分段直接推理]


rtf_avg: 0.355: 100%|██████████| 1/1 [03:27<00:00, 207.62s/it]                                                                                            


    直接推理完成, 输出长度: 194
    CER: 99.5% | ref: 204 | pred: 115
  平均CER=99.9% | 耗时=237.1s


In [13]:
long_ordered = ['SV-base', 'SV-ft', 'Nano-base', 'Nano-ft', 'Nano-base-direct', 'Nano-ft-direct']
long_res_list = [long_all_results[n] for n in long_ordered]

print('=' * 80)
print('长音频六模型对比汇总')
print('=' * 80)
print(f'{"模型":<20} {"平均CER":>8} {"样本":>6} {"平均ref":>8} {"平均pred":>8} {"耗时":>10}')
print('-' * 80)
for r in long_res_list:
    avg_ref = sum(s['ref_len'] for s in r['results']) / max(r['total'], 1)
    avg_pred = sum(s['pred_len'] for s in r['results']) / max(r['total'], 1)
    print(f'{r["name"]:<20} {r["avg_cer"]:>7.1%} {r["total"]:>6} {avg_ref:>8.0f} {avg_pred:>8.0f} {r["time"]:>9.1f}s')
print('=' * 80)

cers = [r['avg_cer'] for r in long_res_list]
sv_imp = (cers[0] - cers[1]) / cers[0] * 100 if cers[0] > 0 else 0
nano_seg = next((r for r in long_res_list if r['name'] == 'Nano-base'), None)
nano_dir = next((r for r in long_res_list if r['name'] == 'Nano-base-direct'), None)
nano_ft_seg = next((r for r in long_res_list if r['name'] == 'Nano-ft'), None)
nano_ft_dir = next((r for r in long_res_list if r['name'] == 'Nano-ft-direct'), None)
if nano_seg and nano_dir:
    print(f'Nano-base 分段→直接 提升: {(nano_seg["avg_cer"]-nano_dir["avg_cer"])/nano_seg["avg_cer"]*100:+.1f}%')
if nano_ft_seg and nano_ft_dir:
    print(f'Nano-ft  分段→直接 提升: {(nano_ft_seg["avg_cer"]-nano_ft_dir["avg_cer"])/nano_ft_seg["avg_cer"]*100:+.1f}%')
print(f'\n注: CER>100% 是因为 pred 远比 ref 长（逐字转录 vs 摘要整理），仅作横向对比')

REPORT_DIR = os.path.expanduser('~/Projects/Agent/local')
long_eval_path = os.path.join(REPORT_DIR, 'long_audio_eval.json')
with open(long_eval_path, 'w', encoding='utf-8') as f:
    json.dump({
        'models': {
            r['name']: {'avg_cer': round(r['avg_cer'], 4), 'total': r['total'],
                        'time': round(r['time'], 1), 'results': r['results']}
            for r in long_res_list
        },
    }, f, ensure_ascii=False, indent=2)
print(f'\n结果已保存: {long_eval_path}')

KeyError: 'Nano-ft-direct'

### [6/8] Nano-ft (长音频, 不分段直接推理)

In [ ]:
print('[6/8] Nano-ft (长音频, 不分段直接推理)')
nano_ft_direct = AutoModel(
    model='FunAudioLLM/Fun-ASR-Nano-2512', hub='ms', disable_update=True, device=device,
    init_param=NANO_CKPT,
)
print('  微调权重已载入')

long_all_results['Nano-ft-direct'] = eval_long_nano(nano_ft_direct, long_samples, 'Nano-ft-direct', direct=True)
print(f'  平均CER={long_all_results["Nano-ft-direct"]["avg_cer"]:.1%} | 耗时={long_all_results["Nano-ft-direct"]["time"]:.1f}s')

del nano_ft_direct
free_memory()